In [1]:
import numpy as np
import networkx as nx
from torch_geometric.datasets import TUDataset


def load_tu_dataset(name: str, root: str = "datasets"):
    dataset = TUDataset(root=root, name=name)

    graphs = []
    y = []

    for data in dataset:
        G = nx.Graph()

        # --- nodes ---
        # TUDataset stores node labels in data.x (one-hot or integer).
        # If x exists, use argmax of one-hot; otherwise fall back to
        # a constant label (some social-network datasets have no node labels).
        num_nodes = data.num_nodes
        for node_idx in range(num_nodes):
            if data.x is not None:
                label = str(data.x[node_idx].argmax().item())
            else:
                label = "0"
            G.add_node(node_idx, feature=label)

        # --- edges ---
        edge_index = data.edge_index.numpy()
        for i in range(edge_index.shape[1]):
            src, dst = int(edge_index[0, i]), int(edge_index[1, i])
            if src < dst:                       # avoid duplicate undirected edges
                G.add_edge(src, dst)

        graphs.append(G)
        y.append(data.y.item())

    y = np.array(y)
    return graphs, y

In [2]:
# Load NCI1
graphs_nci1, y_nci1 = load_tu_dataset("NCI1")
print(f"NCI1: {len(graphs_nci1)} graphs, "
      f"class 0: {(y_nci1 == 0).sum()}, class 1: {(y_nci1 == 1).sum()}")

Processing...
Done!


NCI1: 4110 graphs, class 0: 2053, class 1: 2057


In [3]:
# Load NCI109
graphs_nci109, y_nci109 = load_tu_dataset("NCI109")
print(f"NCI109: {len(graphs_nci109)} graphs, "
      f"class 0: {(y_nci109 == 0).sum()}, class 1: {(y_nci109 == 1).sum()}")

Processing...
Done!


NCI109: 4127 graphs, class 0: 2048, class 1: 2079


In [4]:
# Load all three datasets
datasets = {}
for name in ["AIDS", "Tox21_ARE_training", "Tox21_MMP_training"]:
    graphs, y = load_tu_dataset(name)

    # Remap labels to {-1, +1} to match your existing pipeline
    y = np.where(y == 0, -1, 1)

    n_pos = (y == 1).sum()
    n_neg = (y == -1).sum()
    minority = min(n_pos, n_neg)
    majority = max(n_pos, n_neg)

    print(f"{name:25s} | {len(graphs):,} graphs | "
          f"+1: {n_pos:,}  -1: {n_neg:,} | "
          f"IR: {majority/minority:.1f}:1 | "
          f"minority: {100*minority/len(y):.1f}%")

    datasets[name] = (graphs, y)

Processing...
Done!


AIDS                      | 2,000 graphs | +1: 1,600  -1: 400 | IR: 4.0:1 | minority: 20.0%


Processing...
Done!


Tox21_ARE_training        | 7,167 graphs | +1: 1,098  -1: 6,069 | IR: 5.5:1 | minority: 15.3%


Processing...
Done!


Tox21_MMP_training        | 7,320 graphs | +1: 1,142  -1: 6,178 | IR: 5.4:1 | minority: 15.6%


In [5]:
from torch_geometric.datasets import TUDataset

for name in ["AIDS", "Tox21_ARE_training", "Tox21_MMP_training"]:
    dataset = TUDataset(root="datasets", name=name)
    data = dataset[0]  # peek at the first graph

    print(f"\n{'='*60}")
    print(f"Dataset: {name}")
    print(f"{'='*60}")
    print(f"Num graphs:        {len(dataset)}")
    print(f"Num classes:       {dataset.num_classes}")
    print(f"Num node features: {dataset.num_node_features}")
    print(f"Num edge features: {dataset.num_edge_features}")

    # Node features
    if data.x is not None:
        print(f"\ndata.x shape:      {data.x.shape}")
        print(f"data.x dtype:      {data.x.dtype}")
        print(f"data.x[0]:         {data.x[0]}")
        # Check if one-hot encoded (all values 0 or 1)
        unique_vals = data.x.unique().tolist()
        print(f"Unique values in x: {unique_vals}")
    else:
        print("\ndata.x:            None (no node features)")

    # Edge features
    if data.edge_attr is not None:
        print(f"\ndata.edge_attr shape: {data.edge_attr.shape}")
        print(f"data.edge_attr[0]:    {data.edge_attr[0]}")
    else:
        print("\ndata.edge_attr:    None (no edge features)")

    print(f"\nExample graph: {data.num_nodes} nodes, {data.edge_index.shape[1]//2} edges")


Dataset: AIDS
Num graphs:        2000
Num classes:       2
Num node features: 38
Num edge features: 3

data.x shape:      torch.Size([47, 38])
data.x dtype:      torch.float32
data.x[0]:         tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0.])
Unique values in x: [0.0, 1.0]

data.edge_attr shape: torch.Size([106, 3])
data.edge_attr[0]:    tensor([1., 0., 0.])

Example graph: 47 nodes, 53 edges

Dataset: Tox21_ARE_training
Num graphs:        7167
Num classes:       2
Num node features: 54
Num edge features: 4

data.x shape:      torch.Size([24, 54])
data.x dtype:      torch.float32
data.x[0]:         tensor([0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
Unique values in x: [0.